In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using:", device)

# ============================================
# TRANSFORMS
# ============================================
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408),
                         (0.2675, 0.2565, 0.2761))
])

# ============================================
# LOAD TRAINING DATA (20 classes)
# ============================================
train_data_path = "Training_data 2"

train_dataset = datasets.ImageFolder(train_data_path, transform=train_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print("Number of classes:", len(train_dataset.classes))
print("Classes:", train_dataset.classes)
print("Train samples:", len(train_dataset))


Using: mps
Number of classes: 20
Classes: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '3', '4', '5', '6', '7', '8', '9']
Train samples: 10000


In [3]:
import torch.nn as nn
from torchvision import models

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 20)   # <-- VERY IMPORTANT

model = model.to(device)
print("Model ready for 20 classes")


Model ready for 20 classes


In [7]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using:", device)

# RESET MODEL — FRESH RESNET-18
model = models.resnet18(weights=None)      # no pretrained weights
model.fc = nn.Linear(model.fc.in_features, 20)   # <-- 20 classes
model = model.to(device)

print("✔ Fresh ResNet-18 initialized for 20 classes")


Using: mps
✔ Fresh ResNet-18 initialized for 20 classes


In [9]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_data_path = "Training_data 2"   # folder with 0..19

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408),
                         (0.2675, 0.2565, 0.2761))
])

train_dataset = datasets.ImageFolder(train_data_path, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print("Classes:", train_dataset.classes)
print("Number of samples:", len(train_dataset))


Classes: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '3', '4', '5', '6', '7', '8', '9']
Number of samples: 10000


In [11]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=5e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=25)


In [13]:
from tqdm import tqdm

def train_20class_fresh(model, train_loader, criterion, optimizer, scheduler, device, epochs=25):
    best_acc = 0
    
    for epoch in range(epochs):
        model.train()
        
        total_loss = 0
        total_correct = 0
        total_samples = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            total_correct += (outputs.argmax(1) == labels).sum().item()
            total_samples += labels.size(0)

            pbar.set_postfix({
                "loss": total_loss / total_samples,
                "acc": 100 * total_correct / total_samples
            })

        epoch_loss = total_loss / total_samples
        epoch_acc = 100 * total_correct / total_samples
        
        print(f"Epoch {epoch+1}: Loss={epoch_loss:.4f} Acc={epoch_acc:.2f}%")
        
        # SAVE BEST MODEL ONLY
        if epoch_acc > best_acc:
            best_acc = epoch_acc
            torch.save(model.state_dict(), "best_model_20class.pth")
            print(f"✔ Saved BEST model (accuracy: {best_acc:.2f}%)")

        scheduler.step()

    print("Training finished. Best Accuracy:", best_acc)


In [15]:
train_20class_fresh(model, train_loader, criterion, optimizer, scheduler, device, epochs=25)


Epoch 1/25: 100%|██████████| 157/157 [00:16<00:00,  9.47it/s, loss=2.25, acc=32.2]


Epoch 1: Loss=2.2485 Acc=32.25%
✔ Saved BEST model (accuracy: 32.25%)


Epoch 2/25: 100%|██████████| 157/157 [00:17<00:00,  9.09it/s, loss=1.82, acc=43.6]


Epoch 2: Loss=1.8227 Acc=43.62%
✔ Saved BEST model (accuracy: 43.62%)


Epoch 3/25: 100%|██████████| 157/157 [00:19<00:00,  8.24it/s, loss=1.64, acc=49.4]


...

Epoch 23: Loss=0.4411 Acc=85.36%
✔ Saved BEST model (accuracy: 85.36%)


Epoch 24/25: 100%|██████████| 157/157 [00:17<00:00,  8.87it/s, loss=0.425, acc=86]  


Epoch 24: Loss=0.4255 Acc=86.04%
✔ Saved BEST model (accuracy: 86.04%)


Epoch 25/25: 100%|██████████| 157/157 [00:17<00:00,  8.89it/s, loss=0.418, acc=86.3]


Epoch 25: Loss=0.4181 Acc=86.27%
✔ Saved BEST model (accuracy: 86.27%)
Training finished. Best Accuracy: 86.27


In [ ]:
#from here i will test the data 

In [ ]:
#best_model_20class.pth
#exist

In [19]:
from torchvision import models
import torch.nn as nn
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using:", device)

# reload fresh architecture
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 20)

# load best saved weights
model.load_state_dict(torch.load("best_model_20class.pth", map_location=device))
model = model.to(device)
model.eval()

print("✔ Model loaded successfully")


Using: mps


/var/folders/_8/mjvw012d6mv79z42766yjs6w0000gp/T/ipykernel_73142/1105722903.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_model

✔ Model loaded successfully


In [ ]:
#STEP 2 — Prepare the Transform for TEST IMAGES

In [21]:
from torchvision import transforms

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408),
                         (0.2675, 0.2565, 0.2761))
])


In [ ]:
#STEP 3 — Load ALL 2000 test images in correct order

In [23]:
import os
from PIL import Image
import torch

def load_test_images(folder):
    file_names = sorted(os.listdir(folder))   # ensures correct order
    images = []

    for name in file_names:
        path = os.path.join(folder, name)
        img = Image.open(path).convert("RGB")
        img = test_transform(img)
        images.append(img)

    return torch.stack(images), file_names

test_folder = "411287098testingdataproject2"

imgs, file_names = load_test_images(test_folder)

print("Total test images:", imgs.shape)


Total test images: torch.Size([2000, 3, 32, 32])


In [ ]:
#STEP 4 — Predict and Save to studentID.txt

In [25]:
output_filename = "411287098project2.txt"

imgs = imgs.to(device)

with torch.no_grad():
    outputs = model(imgs)
    _, preds = torch.max(outputs, 1)

preds = preds.cpu().tolist()

# save to file
with open(output_filename, "w") as f:
    for p in preds:
        f.write(str(p) + "\n")

print("✔ Predictions saved to:", output_filename)
print("✔ Total predictions:", len(preds))


✔ Predictions saved to: 411287098project2.txt
✔ Total predictions: 2000


In [27]:
lines = open("411287098project2.txt").read().splitlines()
print("Lines in file:", len(lines))


Lines in file: 2000


In [43]:
output_filename = "411287098project2.txt"

imgs = imgs.to(device)

with torch.no_grad():
    outputs = model(imgs)
    _, preds = torch.max(outputs, 1)

preds = preds.cpu().tolist()

# save with required format
with open(output_filename, "w") as f:
    for i, p in enumerate(preds):
        image_number = str(i+1).zfill(4)   # 0001, 0002, 0003, ...
        f.write(f"{image_number} {p}\n")

print("✔ File saved in correct format:", output_filename)
print("✔ Total predictions:", len(preds))


✔ File saved in correct format: 411287098project2.txt
✔ Total predictions: 2000


In [33]:
with open("411287098.txt") as f:
    for i in range(5):
        print(f.readline().strip())


0001 10
0002 5
0003 2
0004 10
0005 3


In [35]:
import collections
print(collections.Counter(preds))


Counter({0: 111, 13: 107, 9: 107, 19: 107, 7: 106, 1: 104, 2: 103, 12: 103, 16: 103, 10: 102, 5: 101, 17: 101, 15: 100, 4: 98, 18: 95, 11: 95, 8: 93, 6: 92, 3: 89, 14: 83})


In [37]:
import os
print(sorted(os.listdir("Training_data 2")))


['.DS_Store', '0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '3', '4', '5', '6', '7', '8', '9']


In [39]:
train_dataset.class_to_idx


{'0': 0,
 '1': 1,
 '10': 2,
 '11': 3,
 '12': 4,
 '13': 5,
 '14': 6,
 '15': 7,
 '16': 8,
 '17': 9,
 '18': 10,
 '19': 11,
 '2': 12,
 '3': 13,
 '4': 14,
 '5': 15,
 '6': 16,
 '7': 17,
 '8': 18,
 '9': 19}

In [45]:
wrong_mapping = {
 '0': 0, '1': 1, '10': 2, '11': 3, '12': 4, '13': 5,
 '14': 6, '15': 7, '16': 8, '17': 9, '18': 10, '19': 11,
 '2': 12, '3': 13, '4': 14, '5': 15, '6': 16, '7': 17,
 '8': 18, '9': 19
}

# reverse mapping (predicted→correct)
reverse_mapping = {v: int(k) for k, v in wrong_mapping.items()}
reverse_mapping


{0: 0,
 1: 1,
 2: 10,
 3: 11,
 4: 12,
 5: 13,
 6: 14,
 7: 15,
 8: 16,
 9: 17,
 10: 18,
 11: 19,
 12: 2,
 13: 3,
 14: 4,
 15: 5,
 16: 6,
 17: 7,
 18: 8,
 19: 9}

In [47]:
input_file = "411287098.txt"
output_file = "411287098_fixed.txt"

wrong_mapping = {
 '0': 0, '1': 1, '10': 2, '11': 3, '12': 4, '13': 5,
 '14': 6, '15': 7, '16': 8, '17': 9, '18': 10, '19': 11,
 '2': 12, '3': 13, '4': 14, '5': 15, '6': 16, '7': 17,
 '8': 18, '9': 19
}

# reverse mapping: predicted → true
reverse_mapping = {v: int(k) for k, v in wrong_mapping.items()}

fixed_lines = []

with open(input_file, "r") as f:
    for line in f:
        img, pred = line.strip().split()
        pred = int(pred)
        corrected = reverse_mapping[pred]
        fixed_lines.append(f"{img} {corrected}")

with open(output_file, "w") as f:
    f.write("\n".join(fixed_lines))

print("✔ Fixed file saved as:", output_file)


✔ Fixed file saved as: 411287098_fixed.txt
